# 05. [Retrieval 1] OpenAI 임베딩 모델·Chroma 설정 확정
  임베딩 모델 설정 확정 부분

In [1]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

In [ ]:
import time # 속도 측정
from langchain_openai import OpenAIEmbeddings # 임베딩 모델

# 임베딩 모델 생성
models = {
    "OpenAI-Small": OpenAIEmbeddings(model="text-embedding-3-small"),
    "OpenAI-Large": OpenAIEmbeddings(model="text-embedding-3-large")
}

In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.metrics.pairwise import cosine_similarity

# 1. 가상 데이터 로드
chunks = []
with open('../samples/processed/sample_chunks.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        chunks.append(json.loads(line))

texts = [chunk["text"] for chunk in chunks]
query = "공공 AI 학습지원 플랫폼의 총사업예산과 수행 기간은?" # 테스트 질문

# 테스트 로직
results = []
for name, model in models.items():
    # 임베딩 생성 및 시간 측정
    start_time = time.time()
    chunk_embeddings = model.embed_documents(texts)
    query_embedding = model.embed_query(query)
    duration = time.time() - start_time
    
    # 유사도 계산 (가장 높은 점수 확인)
    similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]
    best_idx = np.argmax(similarities)
    
    results.append({
        "Model": name,
        "Time(sec)": round(duration, 4), # 소요시간, 0.0001초단위로 반올림
        "Best_Score": round(similarities[best_idx], 4), # 유사도 점수, 0.0001단위로 반올림
        "Best_Chunk": f"Chunk_{best_idx+1}" # 가장 유사한 Chunk ID
    })


In [ ]:
df_results = pd.DataFrame(results)
print(df_results)

In [ ]:
# Small 모델 토큰 / 비용
from tiktoken import encoding_for_model

enc = encoding_for_model("text-embedding-3-small")
tokens = sum(len(enc.encode(t)) for t in texts)
cost = tokens / 1000 * 0.00002  # small 모델 가격 (구글링))
print(f" Small 모델 토큰 수: {tokens}, 비용: {cost}")

In [ ]:
# large 모델 토큰 / 비용
enc = encoding_for_model("text-embedding-3-large")
tokens = sum(len(enc.encode(t)) for t in texts)
cost = tokens / 1000 * 0.00013  # large 모델 가격 (구글링)
print(f" Large 모델 토큰 수: {tokens}, 비용: {cost}")

In [ ]:
query = "공공 AI 학습지원 플랫폼의 총사업예산과 수행 기간은?"

# 1. 모델 설정 (Best Score가 높은 Small 모델)
model = OpenAIEmbeddings(model="text-embedding-3-small")
query_embedding = model.embed_query(query)
chunk_embeddings = model.embed_documents(texts)

# 2. 유사도 계산
similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]

# 3. 가장 유사도가 높은 청크의 인덱스 찾기
best_idx = np.argmax(similarities)

# 4. 상세 내용 출력
print(f"- 청크 ID: {chunks[best_idx]['chunk_id']}")
print(f"- 유사도 점수: {round(similarities[best_idx], 4)}")
print(f"- 본문 내용:\n{chunks[best_idx]['text']}")

# 06. [Retrieval 1] OpenAI + Chroma 기본 유사도 검색 구현


In [ ]:
import os
import sys
import yaml
# 노트북 폴더(notebook/)에서 상위 루트 경로를 인식시키기 위함
sys.path.append("../")
from src.openai_chroma_retriever import OpenAIChromaRetriever

# 1. 실제 내장된 YAML 파일 읽기
yaml_path = "../config/default.yaml"
with open(yaml_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 잘 로드됐는지 정보 확인 
print(f"-> active_profile: {config['retrieval']['active_profile']}")
print(f"-> 기본 검색 개수(top_k): {config['retrieval']['top_k']}") # yaml에서 수정할때마다 바뀜
print(f"-> 샘플 청크 경로: {config['paths']['chunks']}")

In [ ]:
# 3. 환경변수에 API Key가 잘 들어있는지 최종 확인
if "OPENAI_API_KEY" not in os.environ:
    print(" 경고: OPENAI_API_KEY가 설정되어 있지 않습니다. 검색 시 에러가 발생할 수 있습니다.")

# 4. 리트리버 객체 생성 (전체 config를 그대로 전달)
retriever = OpenAIChromaRetriever(config=config)

# 5. 실제 문서 내용에 있을 법한 키워드로 테스트 질문 투척!
test_query = "사업 제안서 서류 제출 마감 기한과 필수 제출 서류 목록이 어떻게 되나요?" 
print(f"테스트 질문: '{test_query}'\n")

# 6. 검색 실행
results = retriever.search(test_query)

# 7. 출력 결과 확인
print(f"검색된 근거 청크 개수: {len(results)}개 \n") #YAML에 설정된 top_k 만큼 나옵니다
for idx, res in enumerate(results, 1):
    print(f"[{idx}순위 근거]")
    print(f"- chunk_id: {res.chunk_id}")
    print(f"- doc_id: {res.doc_id}")
    print(f"- score (거리 값): {res.score:.4f}")
    print(f"- 본문 텍스트: {res.text}")
    print("-" * 60)

# 07. [Retrieval 1] OpenAI 메타데이터 필터링 구현

In [4]:
import yaml
import os
import sys
sys.path.append("../")
from src.openai_chroma_retriever import OpenAIChromaRetriever

# 1. 설정 로드
yaml_path = "../config/default.yaml"
with open(yaml_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 2. 리트리버 초기화
retriever = OpenAIChromaRetriever(config=config)
test_query = "사업명이 뭐야?"

In [5]:
print("== 필터 없이 기본 검색 ==")

results = retriever.search(query=test_query)
for r in results:
    print(f"- [score: {r.score:.2f}] agency: {r.metadata.get('agency')}")
    print(f"  [본문 미리보기]: {r.text[:150]}...\n")

== 필터 없이 기본 검색 ==


- [score: 1.47] agency: 가상디지털진흥원
  [본문 미리보기]: [가상 제안요청서] 이 문서는 RAG 시스템 개발과 검색 성능 실험을 위해 작성한 가상 데이터입니다. 실제 기관, 사업 또는 입찰 공고와 관련이 없습니다. 1. 사업 개요 사업명은 "2026년 공공 AI 학습지원 플랫폼 구축 사업"이며 발주기관은 가상디지털진흥원이다. ...

- [score: 1.50] agency: 가상디지털진흥원
  [본문 미리보기]: REST API 방식으로 연계하고, 매일 오전 2시에 변경된 사용자 정보를 동기화해야 한다. 데이터 이전 결과는 전체 건수, 성공 건수, 실패 건수로 구분한 검증 보고서로 제출해야 한다. 4. 성능 및 품질 요구사항 동시 접속자 1,000명 환경에서 일반 화면의 평균 ...

- [score: 1.50] agency: 가상디지털진흥원
  [본문 미리보기]: 이해도, 수행 방안, 인력 구성, 데이터 이전 계획, 보안 대책이 포함된다. 7. 산출물 착수계획서, 요구사항정의서, 화면설계서, API 명세서, 데이터 이전 결과서, 시험 결과서, 관리자 및 사용자 매뉴얼, 보안 점검 결과서, 완료보고서를 제출해야 한다. 모든 산출물...



현재 샘플 데이터의 청크 3개 모두 agency가 가상디지털진흥원이기에 결과는 이렇게 나옴 

In [12]:
print("== agency 검색 ==")

results_agency = retriever.search(query=test_query, filters={"agency": "     가상디지털진흥원 "})

for res in results_agency:
    print(f"- [score: {res.score:.2f}] agency: {res.metadata.get('agency')}")

== agency 검색 ==


- [score: 1.47] agency: 가상디지털진흥원
- [score: 1.50] agency: 가상디지털진흥원
- [score: 1.50] agency: 가상디지털진흥원


strip()을 추가하여 앞뒤 띄어쓰기가 있어도 검색 됨

In [7]:
print("== agency 부분 검색 ==")

results_agency = retriever.search(query=test_query, filters={"agency": " 가상 "})

for res in results_agency:
    print(f"- [score: {res.score:.2f}] agency: {res.metadata.get('agency')}")

== agency 부분 검색 ==
- [score: 1.47] agency: 가상디지털진흥원
- [score: 1.50] agency: 가상디지털진흥원
- [score: 1.50] agency: 가상디지털진흥원


"가상" 으로만 검색해도 가상디지털진흥원의 부분 일치된 청크가 검색되는 모습

In [8]:
print("== doc_id 정확도 필터 ('sample_rfp')==")

results_doc = retriever.search(query=test_query, filters={"doc_id": "sample_rfp"})

for res in results_doc:
    print(f"- [score: {res.score:.2f}] doc_id: {res.doc_id}")

== doc_id 정확도 필터 ('sample_rfp')==
- [score: 1.47] doc_id: sample_rfp
- [score: 1.50] doc_id: sample_rfp
- [score: 1.50] doc_id: sample_rfp


현재 청크된 샘플 데이터 모두 sample_rfp가 doc_id 기에 결과는 이렇게

In [9]:
print("== 필터 미일치 (존재하지 않는 기관) ==")

results_empty = retriever.search(query=test_query, filters={"agency": "우주항공청"})

print(f"결과 개수: {len(results_empty)}개")

== 필터 미일치 (존재하지 않는 기관) ==
안내: '{'agency': '우주항공청'}' 조건에 일치하는 문서를 찾을 수 없습니다.
결과 개수: 0개


우주 항공청이라는 agency는 샘플에 없으니 결과는 0개

In [17]:
print("== 사업명 검색 ==")

results_title = retriever.search(query=test_query, filters={"title": "2026년 공공 AI 학습지원 플랫폼"})

for res in results_title:
    project_title = res.metadata.get("title", "N/A")
    print(f"- [score: {res.score:.2f}] doc_id: {res.doc_id} | title: {project_title}")

== 사업명 검색 ==
- [score: 1.47] doc_id: sample_rfp | title: 2026년 공공 AI 학습지원 플랫폼 구축 사업
- [score: 1.50] doc_id: sample_rfp | title: 2026년 공공 AI 학습지원 플랫폼 구축 사업
- [score: 1.50] doc_id: sample_rfp | title: 2026년 공공 AI 학습지원 플랫폼 구축 사업
